# PureVQ-GAN directly on BrainWash — one-day optimisation plan
No paper-setting validation (your call): we train a handful of purifier variants, **pre-screen them in seconds** on the
real BrainWash noise (how much perturbation survives vs. how much they blur clean images), and spend the expensive
stage-4 runs only on the two best.

| block | what | A100 time (estimate — check the `sec` field of the first epochs and adjust `EPOCHS`) |
|---|---|---|
| 1 | train 4 purifier variants (CIFAR-10 / task-data, K, latent size) | ~2–4 h total |
| 2 | pre-screen all variants × 1/2 passes on ε=0.1 and ε=0.3 noise | ~5 min |
| 3 | undefended references (shared with the EBM notebook) | 2 × 10–30 min |
| 4 | stage 4 for the top-2: clean+PVQ (utility) and cautious ε=0.1+PVQ | 4 × 10–30 min |
| 5 | winner on ε=0.3 | 2 × 10–30 min |

Variants (why each): **A** paper default (K=512, 8×8 latent, generic CIFAR-10 images); **B** smaller codebook K=128 (stronger
bottleneck); **C** 4×4 latent (much stronger spatial bottleneck); **D** trained on the poisoned task data itself (the PureVQ-GAN
paper's own protocol). Selection rule (fixed before running): among variants whose clean PSNR is within 3 dB of the best, take the
2 with the lowest perturbation survival on ε=0.1; operating point = best recovery with clean+PVQ accuracy within 3 points of clean.

**Before you start:** on the cluster, pack the noise you already trained and put it on Google Drive:
```bash
cd ~/Brainwash && tar czf bw_noise_ewc.tgz -C repro/sweep/noise cifar100_ewc_cautious_eps0.1 cifar100_ewc_cautious_eps0.3
```
Download `bw_noise_ewc.tgz`, upload it to `MyDrive/`, then run the next cell once to unpack it to `MyDrive/bw_noise/`.
(The noise pkl already contains the victim checkpoint, so nothing else from the cluster is needed.)

In [ ]:
import os, sys, json, glob
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/brainwash-repro'
if not os.path.exists(REPO):
    !git clone -b claude/adoring-clarke-97rxkd https://github.com/anushkathagle/brainwash-repro.git $REPO
else:
    !git -C $REPO pull --ff-only
%cd $REPO
!pip -q install huggingface_hub safetensors
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU (A100/L4 strongly preferred)'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
sys.path.insert(0, f'{REPO}/repro/defense')
import colab_utils as cu

In [ ]:
if os.path.exists('/content/drive/MyDrive/bw_noise_ewc.tgz') and not os.path.exists('/content/drive/MyDrive/bw_noise'):
    os.makedirs('/content/drive/MyDrive/bw_noise')
    !tar xzf /content/drive/MyDrive/bw_noise_ewc.tgz -C /content/drive/MyDrive/bw_noise
!ls -la /content/drive/MyDrive/bw_noise/*

In [ ]:
# ---- where your BrainWash noise lives (copy repro/sweep/noise/<tag>/ folders from the cluster to Drive) ----
NOISE_ROOT = '/content/drive/MyDrive/bw_noise'            # contains cifar100_ewc_cautious_eps0.1/<file>.pkl etc.
RESULTS    = '/content/drive/MyDrive/bw_defense_results'  # shared by both notebooks -> reference runs happen once
METHOD     = 'ewc'
TAG_01     = f'cifar100_{METHOD}_cautious_eps0.1'
TAG_03     = f'cifar100_{METHOD}_cautious_eps0.3'
PKL = {t: cu.find_pkl(NOISE_ROOT, t) for t in (TAG_01, TAG_03)}
print(json.dumps(PKL, indent=1))

PUR_ROOT = '/content/drive/MyDrive/bw_purifiers'
EPOCHS = 60        # paper: 100. Lower it if the first epochs show the full run won't fit your day.
VARIANTS = {
  'A_c10_K512_L8': f'--dataset cifar10 --num_codes 512 --latent_size 8 --epochs {EPOCHS}',
  'B_c10_K128_L8': f'--dataset cifar10 --num_codes 128 --latent_size 8 --epochs {EPOCHS}',
  'C_c10_K512_L4': f'--dataset cifar10 --num_codes 512 --latent_size 4 --epochs {EPOCHS}',
  # D is trained per noise file (the paper trains on the poisoned set itself); 5,000 images -> minutes
  'D_task01_K512_L8': f'--dataset "bw_poisoned:{PKL[TAG_01]}" --bw_experiment split_cifar100 --num_codes 512 --latent_size 8 --epochs 100',
  'D_task03_K512_L8': f'--dataset "bw_poisoned:{PKL[TAG_03]}" --bw_experiment split_cifar100 --num_codes 512 --latent_size 8 --epochs 100',
}

## Block 1 — train purifiers (resumable: re-run after a disconnect)

In [ ]:
for name, a in VARIANTS.items():
    out = f'{PUR_ROOT}/{name}'
    if os.path.exists(f'{out}/purifier.pt'):
        print('[done]', name); continue
    cu.sh(f'python -m purevqgan.train {a} --img_size 32 --data_root ./data --amp --resume --out {out}')
for name in VARIANTS:
    last = open(f'{PUR_ROOT}/{name}/train_log.jsonl').read().strip().splitlines()[-1]
    print(name, json.loads(last))

## Block 2 — pre-screen (seconds per variant): perturbation survival vs. clean distortion

In [ ]:
rows = []
for name in VARIANTS:
    for tag in (TAG_01, TAG_03):
        if name.startswith('D_task01') and tag != TAG_01: continue
        if name.startswith('D_task03') and tag != TAG_03: continue
        for passes in (1, 2):
            j = f'{RESULTS}/prescreen/{name}_{tag}_p{passes}.json'
            if not os.path.exists(j):
                os.makedirs(os.path.dirname(j), exist_ok=True)
                cu.sh(f'python -m purevqgan.purify --purifier {PUR_ROOT}/{name}/purifier.pt --experiment split_cifar100 '
                      f'--noise_pkl "{PKL[tag]}" --passes {passes} --json {j} > /dev/null')
            r = json.load(open(j)); rows.append(dict(variant=name, tag=tag, passes=passes, **{k: r.get(k) for k in
                ('survival_l2', 'code_flip', 'psnr_clean', 'psnr_poisoned', 'input_psnr_poisoned')}))
import pandas as pd
df = pd.DataFrame(rows); display(df.round(3))
e01 = df[df.tag == TAG_01]
cand = e01[e01.psnr_clean >= e01.psnr_clean.max() - 3].sort_values('survival_l2').head(2)
TOP = list(cand[['variant', 'passes']].itertuples(index=False, name=None))
print('selected for stage 4:', TOP)

## Block 3 — undefended references

In [ ]:
# undefended references IN THIS ENVIRONMENT (GPU/torch differ from the cluster; ~10-30 min each; skipped if done)
ref_clean = cu.stage4(f'{METHOD}_clean__undefended',        PKL[TAG_01], 'clean', METHOD, RESULTS)
ref_01    = cu.stage4(f'{METHOD}_cautious0.1__undefended',  PKL[TAG_01], 'ours',  METHOD, RESULTS)
cu.table([ref_clean, ref_01])
print('cluster reference (RESULTS_cifar100.md, EWC): clean -5.2 / 68.4 ; cautious eps0.1 -9.5 / 56.4 ; cautious eps0.3 -24.7 / 43.2')

## Block 4 — stage 4 for the top-2 variants (ε=0.1)

In [ ]:
runs = {}
for name, passes in TOP:
    d = f'--purifier {PUR_ROOT}/{name}/purifier.pt --purify_passes {passes}'
    runs[(name, passes)] = (cu.stage4(f'{METHOD}_clean__pvq_{name}_p{passes}',       PKL[TAG_01], 'clean', METHOD, RESULTS, d),
                            cu.stage4(f'{METHOD}_cautious0.1__pvq_{name}_p{passes}', PKL[TAG_01], 'ours',  METHOD, RESULTS, d))
cu.table([ref_clean, ref_01] + [r for p in runs.values() for r in p], undef=ref_01, clean=ref_clean)
ok = {k: cu.recovery(p[1]['bwt'], ref_01['bwt'], ref_clean['bwt']) for k, p in runs.items() if ref_clean['acc'] - p[0]['acc'] <= 3}
BEST = max(ok, key=ok.get) if ok else None
print('within utility margin:', ok, '\noperating point:', BEST)

## Block 5 — winner on ε=0.3

In [ ]:
assert BEST, 'no variant kept clean accuracy within 3 points; report the trade-off from block 4 instead'
name, passes = BEST
if name.startswith('D_task01'): name = name.replace('task01', 'task03')   # task-data purifier must match the noise file
d = f'--purifier {PUR_ROOT}/{name}/purifier.pt --purify_passes {passes}'
ref_03 = cu.stage4(f'{METHOD}_cautious0.3__undefended', PKL[TAG_03], 'ours', METHOD, RESULTS)
def_03 = cu.stage4(f'{METHOD}_cautious0.3__pvq_{name}_p{passes}', PKL[TAG_03], 'ours', METHOD, RESULTS, d)
cu.table([ref_03, def_03], undef=ref_03, clean=ref_clean)

**If you have time left:** try `--base_ch 256 --num_res_blocks 2` (~58M params) for the winning variant, or `EPOCHS = 100`.
Record every attempt — the full table is what goes in the thesis, not just the best row.